# B1 & B2 — Predictions (committed BEFORE running)

## B1 — Parameter count by hand

A Dense(n) layer receiving an m-dimensional input has n*(m+1) parameters (weights + biases).

| Layer | Input | Output | Weights | Biases | Total params |
|---|---|---|---|---|---|
| Flatten | 32x32x3 | 3072 | - | - | 0 |
| Dense(200, relu) | 3072 | 200 | 614,400 | 200 | **614,600** |
| Dense(150, relu) | 200 | 150 | 30,000 | 150 | **30,150** |
| Dense(10, softmax) | 150 | 10 | 1,500 | 10 | **1,510** |
| **Total** | | | | | **646,260** |

**Dominant layer:** Dense(200) carries 614,600 / 646,260 = **95.1%** of all parameters.

**Why:** it sits immediately after Flatten, so its input is the full 3072-dimensional raw pixel vector. Every pixel connects to every one of its 200 units. Later layers connect only hundreds of units, so their cost is negligible. This is exactly the parameter tax that convolutional layers are designed to avoid.

## B2 — Expected performance

**Random guessing on CIFAR-10:** 10 equally likely classes -> **10%**.

**Predicted test accuracy:** **~40-45%**, most likely near **0.42**.

**Reasoning:**
- Foster, Ch. 2 reports ~40% test accuracy for an MLP of this size on CIFAR-10.
- Flatten destroys spatial structure, so the network cannot learn translation-invariant features (edges, textures, object parts).
- 10 epochs at lr=0.0005 with Adam converges but plateaus early.
- 646k parameters trained on 50k images with no dropout, no batchnorm, no augmentation -> overfitting expected: train accuracy should exceed test accuracy noticeably.


# 👀 Multilayer perceptron (MLP)

In this notebook, we'll walk through the steps required to train your own multilayer perceptron on the CIFAR dataset

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from tensorflow.keras import layers, models, optimizers, utils, datasets
from notebooks.utils import display

## 0. Parameters <a name="parameters"></a>

In [ ]:
NUM_CLASSES = 10

## 1. Prepare the Data <a name="prepare"></a>

In [ ]:
(x_train, y_train), (x_test, y_test) = datasets.cifar10.load_data()

In [ ]:
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

y_train = utils.to_categorical(y_train, NUM_CLASSES)
y_test = utils.to_categorical(y_test, NUM_CLASSES)

In [ ]:
display(x_train[:10])
print(y_train[:10])

## 2. Build the model <a name="build"></a>

In [ ]:
input_layer = layers.Input((32, 32, 3))

x = layers.Flatten()(input_layer)
x = layers.Dense(200, activation="relu")(x)
x = layers.Dense(150, activation="relu")(x)

output_layer = layers.Dense(NUM_CLASSES, activation="softmax")(x)

model = models.Model(input_layer, output_layer)

model.summary()

## 3. Train the model <a name="train"></a>

In [ ]:
opt = optimizers.Adam(learning_rate=0.0005)
model.compile(
    loss="categorical_crossentropy", optimizer=opt, metrics=["accuracy"]
)

In [ ]:
model.fit(x_train, y_train, batch_size=32, epochs=10, shuffle=True)

## 4. Evaluation <a name="evaluate"></a>

In [ ]:
model.evaluate(x_test, y_test)

In [ ]:
CLASSES = np.array(
    [
        "airplane",
        "automobile",
        "bird",
        "cat",
        "deer",
        "dog",
        "frog",
        "horse",
        "ship",
        "truck",
    ]
)

preds = model.predict(x_test)
preds_single = CLASSES[np.argmax(preds, axis=-1)]
actual_single = CLASSES[np.argmax(y_test, axis=-1)]

In [ ]:
n_to_show = 10
indices = np.random.choice(range(len(x_test)), n_to_show)

fig = plt.figure(figsize=(15, 3))
fig.subplots_adjust(hspace=0.4, wspace=0.4)

for i, idx in enumerate(indices):
    img = x_test[idx]
    ax = fig.add_subplot(1, n_to_show, i + 1)
    ax.axis("off")
    ax.text(
        0.5,
        -0.35,
        "pred = " + str(preds_single[idx]),
        fontsize=10,
        ha="center",
        transform=ax.transAxes,
    )
    ax.text(
        0.5,
        -0.7,
        "act = " + str(actual_single[idx]),
        fontsize=10,
        ha="center",
        transform=ax.transAxes,
    )
    ax.imshow(img)